In [1]:
"""Main file for running annealed Langevin dynamics for new material sampling."""
from __future__ import annotations
from pathlib import Path
from types import SimpleNamespace

import torch

from chggen.pl_data.dataset import CHGNetDataset
from chggen.pl_modules.model_egnn import CHGGen
from chggen.common.data_utils import get_scaler

/home/xzdai/anaconda3/envs/chggen/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [15]:
dataset = CHGNetDataset(
    path='/home/xzdai/ceder_group/material_dircovery/chggen_fully/data/perov_5/test_zpc.csv',
    name = 'A_good_name',
    prop_list = ['heat_all'],
)

100%|██████████| 50/50 [00:00<00:00, 285.54it/s]


In [16]:
lattice_scaler = get_scaler(dataset= dataset)

model_hparams = {'latent_dim': 64, 'hidden_dim': 128, 
                'predict_property': True, 'property_dim': 1, # predict the multiple property 
                'load_pretrain': True, 'fc_num_layers': 1, 
                'sigma_F_begin': 0.5, 'sigma_F_end': 0.005, 
                'sigma_L_begin': 0.5, 'sigma_L_end': 0.005, 
                'type_sigma_begin': 5.0, 'type_sigma_end': 0.01,
                'max_atoms': 10,        # should be larger than the training set.
                'num_noise_level': 100, 
                'lattice_scale_method': 'scale_length', 
                'cost_natom': 1.0, 'cost_latt': 10.0, 'cost_coord': 10.0, 'cost_type': 1.0, 'cost_lattice': 10.0, 'cost_composition': 1.0, 'cost_edge': 10.0, 'cost_property': 1.0,
                'beta': 0.01,
                'teacher_forcing_lattice': True,
                'teacher_forcing_max_epoch': 1000,
                'decoder': 'egnn'}

chggen = CHGGen(
    hparams_dict = model_hparams, lattice_scaler = lattice_scaler, 
)

device = torch.device('cuda')
checkpoint_path = "./test_models/perov/epoch=3.ckpt"
chggen = chggen.load_from_checkpoint(checkpoint_path = checkpoint_path)
chggen.lattice_scaler = lattice_scaler
chggen.to(device = device)
print('Model loaded')

/home/xzdai/ceder_group/material_dircovery/chggen_fully/chggen/common/data_utils.py:619: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  X = torch.tensor(X, dtype=torch.float)


CHGNet initialized with 400,438 parameters
CHGNet initialized with 400,438 parameters
CHGNet initialized with 400,438 parameters
CHGNet initialized with 400,438 parameters
Model loaded


In [24]:
ld_kwargs = SimpleNamespace(
    n_step_each = 100,
    step_lr = 5e-7,
    min_sigma = 0,
    save_traj = False,
    disable_bar = False,
    compute_force = True,
    beta_c = 0,         # property update rate
    beta_f = 0,         # atomic force update rate                          
)
ld_kwargs.n_step_each

100

In [25]:
z = torch.randn(1, 64, requires_grad= False, device = device)

results = chggen.langevin_dynamics_guidance(
    z = z, 
    prop_guidance = torch.tensor(-0.05, device= device), 
    ld_kwargs= ld_kwargs
)

100%|██████████| 50/50 [00:51<00:00,  1.04s/it]


In [26]:
lattices = results['lattices']
num_atoms = results['num_atoms']
frac_coords = results['frac_coords']
atom_types = results['atom_types']

print(lattices)
print(num_atoms)
print(frac_coords)
print(atom_types)

tensor([[[ 2.4435, -2.8813, -1.0349],
         [-1.0049,  0.5409, -3.8715],
         [ 2.9330,  2.5945, -0.4514]]], device='cuda:0')
tensor([5], device='cuda:0')
tensor([[0.6522, 0.2476, 0.5761],
        [0.8823, 0.6185, 0.5802],
        [0.0996, 0.5787, 0.5895],
        [0.8279, 0.0468, 0.0774],
        [0.4200, 0.6486, 0.0783]], device='cuda:0')
tensor([ 9,  8,  7,  8, 41], device='cuda:0')


In [23]:
from pymatgen.core import Structure

In [8]:
ids = [0]+torch.cumsum(num_atoms, dim=0).tolist()

In [9]:
for i in range(len(num_atoms)):
    
    s = Structure(
        lattice = lattices[i].detach().cpu().numpy(), 
        species = atom_types[ids[i]:ids[i+1]].detach().cpu().numpy(), 
        coords = frac_coords[ids[i]:ids[i+1]].detach().cpu().numpy(),
        to_unit_cell = False,
        coords_are_cartesian = False,
    )

    s.to(filename = f'test_models/DiffCSP_structures/structure_{i}.cif')